# Setup

## File download

In [ ]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
import os
from rich import print
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
!pip install pyspark -qq

trip_data_path = "data/raw/yellow_tripdata_2024-01.parquet"
os.makedirs('data/raw', exist_ok=True)
if os.path.exists(trip_data_path):
    print("File already exists, skipping download.")
else:
    response = requests.get("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet", stream=True)
    response.raise_for_status()
    with open(trip_data_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=4194304):  # 4MB chunks to avoid memory issues and speed up download
            f.write(chunk)
    print ("File downloaded")
import socket
print(socket.gethostname())
print(socket.gethostbyname(socket.gethostname()))

## Setting up PySpark

In [ ]:
import pyspark
import subprocess
import time
def check_java():
    try:
        result = subprocess.run(['java', '-version'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if result.returncode == 0:
            print("Java is installed and accessible.")
            print(result.stderr)  # Java version info is usually in stderr
        else:
            print("Java is not accessible. Please check your JAVA_HOME and PATH settings.")
            print(result.stderr)
    except FileNotFoundError:
        print("Java executable not found. Please ensure Java is installed and JAVA_HOME is set correctly.")
from pyspark.sql import SparkSession
os.environ["JAVA_HOME"] = "C:/Program Files/Java/jdk-17" # Update this path to your actual Java installation directory
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.showConsoleProgress=true pyspark-shell"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = "C:\\hadoop"
check_java()
print ("Environment variables set for Spark")
spark = SparkSession.builder \
    .appName("Yellow Taxi Analysis") \
    .master("local[*]") \
    .config("spark.ui.enabled", "false")\
    .config("spark.hadoop.io.native.lib.available", "false") \
    .getOrCreate() # Disable Spark UI and native library check for better performance on local machine
print("Spark started on version:", spark.version)
spark_start_time = time.perf_counter()
df = spark.read.parquet(trip_data_path); spark_end_time = time.perf_counter()
spark_time = spark_end_time - spark_start_time
print(f"Time taken for Spark to read Parquet file: {spark_time:.2f} seconds")
df.show(5)
print ("DataFrame schema:")
df.printSchema()
starting_rows = df.count()
print ("DataFrame row count:", starting_rows)
print ("DataFrame columns:", df.columns)


## Comparing with Pandas

In [ ]:
import pandas as pd
pandas_start_time = time.perf_counter()
p_df = pd.read_parquet(trip_data_path)
pandas_end_time = time.perf_counter()
pandas_time = pandas_end_time - pandas_start_time
print(f"Time taken to read Parquet file with Pandas: {pandas_time:.2f} seconds")
print(f"Spark was {pandas_time / spark_time:.2f} times faster than Pandas for reading the Parquet file. Pandas eagerly loaded the data into memory while Spark lazily loaded the data, which is why Spark was faster for this operation. In conclusion, Spark's lazy loading and optimized execution engine make it more efficient for handling large datasets compared to Pandas, which loads the entire dataset into memory at once. Pandas is trash for big data, Spark is the way to go.")

# Data Cleaning & Feature Engineering in Spark

## Clean the data

In [ ]:
# Cleanup Spark dataframes.
# Removing rows with null values in critical columns (pickup/dropoff times, locations, fare,distance)
df = df.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime",  "fare_amount", "trip_distance", "PULocationID", "DOLocationID"])
print (f"{starting_rows - df.count()} rows removed due to null values. Remaining rows: {df.count()}")
starting_rows = df.count()
# Removing rows with non-positive fare amounts or trip distances
df = df.filter((df.fare_amount > 0) & (df.trip_distance > 0))
print (f"{starting_rows - df.count()} rows removed due to non-positive fare amounts or trip distances. Remaining rows: {df.count()}")
starting_rows = df.count()
starting_rows = df.count()
# Removing rows with pickup times after dropoff times
df = df.filter(df.tpep_pickup_datetime < df.tpep_dropoff_datetime)
print (f"{starting_rows - df.count()} rows removed due to pickup times after dropoff times. Remaining rows: {df.count()}")
starting_rows = df.count()
# Remove rows with ludicrous fares (e.g., fare_amount > $500)
df = df.filter(df.fare_amount <= 500)
print (f"{starting_rows - df.count()} rows removed due to ludicrous fares. Remaining rows: {df.count()}")


## Derive columns

In [ ]:
# We will now derive new columns for analysis. We will derive trip duration in minutes, speed in miles per hour, pickup hour of day, and pickup day of week. Tip percentage will also be derived for later analysis.
from pyspark.sql import functions as F

df = df.withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("tpep_dropoff_datetime")
        - F.unix_timestamp("tpep_pickup_datetime")
    )
    / 60,
)
df = df.withColumn(
    "speed_mph", F.col("trip_distance") / (F.col("trip_duration_minutes") / 60)
)
df = df.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
df = df.withColumn("pickup_dayOfWeek", F.dayofweek("tpep_pickup_datetime"))
df = df.withColumn("tip_percentage", (F.col("tip_amount") / F.col("fare_amount")) * 100)
print(
    "Derived new columns: trip_duration_minutes, speed_mph, pickup_hour, pickup_dayOfWeek, tip_percentage"
)
df.show(5)

## Spark SQL Analytics

In [ ]:
# Register the DataFrame as a temporary view for Spark SQL queries
df.createOrReplaceTempView("trips")
queries = {}
# What are the top 10 busiest pickup hours, and what is the average fare and tippercentage for each?
queries[
    "top_pickup_hours"
] = """
SELECT pickup_hour, COUNT(*) AS trip_count, ROUND(AVG(fare_amount), 2) AS avg_fare, ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
FROM trips
GROUP BY pickup_hour
ORDER BY trip_count DESC
LIMIT 10
"""

# Which day of the week has the highest average trip speed? Include averagedistance and duration
queries[
    "fastest_day_of_week"
] = """
SELECT pickup_dayOfWeek, ROUND(AVG(speed_mph), 2) AS avg_speed, ROUND(AVG(trip_distance), 2) AS avg_distance, ROUND(AVG(trip_duration_minutes), 2) AS avg_duration
FROM trips
GROUP BY pickup_dayOfWeek
ORDER BY avg_speed DESC
LIMIT 1
"""

# Using a window function, rank the top 5 pickup locations by total revenue foreach day of the week.
queries[
    "top_pickup_locations_by_revenue"
] = """
SELECT *
FROM (
    SELECT 
        pickup_dayOfWeek,
        PULocationID,
        ROUND(SUM(fare_amount), 2) AS total_revenue,
        RANK() OVER (
            PARTITION BY pickup_dayOfWeek 
            ORDER BY SUM(fare_amount) DESC
        ) AS revenue_rank
    FROM trips
    GROUP BY pickup_dayOfWeek, PULocationID
)
WHERE revenue_rank <= 5
ORDER BY pickup_dayOfWeek, revenue_rank;
"""

# Calculate the cumulative trip count by hour of day (running total from hour 0 to23). At what hour does the cumulative count surpass 50% of daily trips?
queries[
    "cumulative_trip_count_by_hour"
] = """
WITH hourly_counts AS (
    SELECT pickup_hour, COUNT(*) AS trip_count
    FROM trips
    GROUP BY pickup_hour
),
cumulative_counts AS (
    SELECT pickup_hour, ROUND(SUM(trip_count) OVER (ORDER BY pickup_hour), 2) AS cumulative_trip_count
    FROM hourly_counts
)
SELECT pickup_hour
FROM cumulative_counts
WHERE cumulative_trip_count > (SELECT SUM(trip_count) FROM hourly_counts) * 0.5
ORDER BY pickup_hour
LIMIT 1
"""

# Compare average fare, distance, and tip percentage between short trips (<2miles), medium trips (2–10 miles), and long trips (>10 miles). Which category has thehighest tip percentage?
queries[
    "trip_length_comparison"
] = """
WITH categorized AS (
    SELECT *,
        CASE
            WHEN trip_distance < 2 THEN 'Short'
            WHEN trip_distance BETWEEN 2 AND 10 THEN 'Medium'
            ELSE 'Long'
        END AS trip_length_category
    FROM trips
)
SELECT
    trip_length_category,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
FROM categorized
GROUP BY trip_length_category
ORDER BY avg_tip_percentage DESC
"""

# Execute the queries and print results
for query_name, query in queries.items():
    print(f"\nResults for query: {query_name}")
    result_df = spark.sql(query)
    result_df.show()

## Performance Optimization

In [ ]:
# We want to compare time taken on uncached queries vs cached queries. We will run the top pickup hours query uncached and then cached to compare times.

# Uncached query
uncached_start_time = time.perf_counter()
spark.sql(queries["top_pickup_hours"]).show()
uncached_end_time = time.perf_counter()
uncached_time = uncached_end_time - uncached_start_time

# Cache the DataFrame and run the same query again
df.cache()
before_cached_start_time = time.perf_counter()
spark.sql(queries["top_pickup_hours"]).show()
before_cached_end_time = time.perf_counter()
initial_cached_time = before_cached_end_time - before_cached_start_time
after_cached_start_time = time.perf_counter()
spark.sql(queries["top_pickup_hours"]).show()  # Run the query again to ensure it's cached
after_cached_end_time = time.perf_counter()
final_cached_time = after_cached_end_time - after_cached_start_time
print(f"Time taken for uncached query: {uncached_time:.2f} seconds")
print(f"Time taken for cached query (first run): {initial_cached_time:.2f} seconds")
print(f"Time taken for cached query (second run): {final_cached_time:.2f} seconds")
if initial_cached_time > uncached_time:
    print(f"The first run of the cached query was {initial_cached_time:.2f}/{uncached_time:.2f} = {initial_cached_time / uncached_time:.2f} times slower than the uncached query, which is expected as the first run involves reading from disk and caching the data. However, the second run of the cached query was {final_cached_time:.2f}/ {uncached_time:.2f} = {uncached_time / final_cached_time:.2f} times faster than the uncached query, demonstrating the significant performance improvement after caching.")
else:
    print(f"Caching the DataFrame resulted in a speedup of {uncached_time / final_cached_time:.2f} times for the top pickup hours query. Due to the way Spark optimizes execution, the first run of the query will be slower as it reads from disk and processes the data. Once cached, subsequent runs of the same query will be much faster as Spark can read from memory instead of disk, demonstrating the performance benefits of caching in Spark for repeated queries on the same dataset.")


In [ ]:
#FIXME - writing does not work on windows for some reason
# # Write the cleaned data back as partioned
# df.write.mode("overwrite").partitionBy("pickup_hour").parquet("data/cleaned/yellow_tripdata_2024-01_partitioned.parquet")
# print("Cleaned data written back to disk, partitioned by pickup_hour. We will now read hour 17 data to see how much faster it is to read a partitioned file vs the original unpartitioned file.")
# # Read the partitioned data for hour 17
# partitioned_start_time = time.perf_counter()
# hour_17_df = spark.read.parquet("data/cleaned/yellow_tripdata_2024-01_partitioned.parquet/pickup_hour=17")
# partitioned_end_time = time.perf_counter()
# partitioned_time = partitioned_end_time - partitioned_start_time
# print(f"Time taken to read partitioned data for hour 17: {partitioned_time:.2f} seconds")
# print(f"Time taken to read the original unpartitioned Parquet file: {spark_time:.2f} seconds")
# print(f"Reading the partitioned data for hour 17 was {spark_time / partitioned_time:.2f} times faster than reading the original unpartitioned Parquet file. This demonstrates the performance benefits of partitioning in Spark, as it allows Spark to read only the relevant subset of data (hour 17) instead of scanning the entire dataset, resulting in significantly faster read times for queries that filter on the partition column.")

# RAG Pipeline over Transportation Documents

## Read the documents into memory

In [ ]:
import PyPDF2
files_memory = {}
for file in os.listdir("docs/"):
    files_memory[file] = PyPDF2.PdfReader(os.path.join("docs/", file))
print ("Files read:",files_memory.keys())
for path in files_memory.values():
    for page in path.pages:
        print(f'{file} - Page {page}')
        print (page.extract_text())
        print ("----- End of page -----\n")

## Chunk the documents

Defining a reusable function

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sympy import re
def clean_text(text):
    # Remove many repeated . which can cause issues with embedding models and chunking
    # text = re.sub(r'\.{2,}', '.', text) TODO: This is a bit too aggressive and can cause issues with some documents. We will remove this for now and see if it causes any issues. If it does, we can reintroduce it with a more nuanced approach.
    return " ".join(text.split())  # removes weird spacing + newlines
    #Remove segments that are not helpful for embedding models, such as page numbers, headers, footers, etc. This is a simple heuristic and can be improved with more sophisticated methods if needed.
    # For example, we can remove lines that are just numbers (page numbers) or lines that are too short (headers/footers). This is a very basic cleaning function and can be further enhanced based on the specific structure of the documents being processed.
    if text.has ("CONTENTS") or text.has ("TABLE OF CONTENTS") or text.has ("INDEX") or text.has ("REFERENCES") or text.has ("BIBLIOGRAPHY") or text.has ("ACKNOWLEDGEMENTS") or text.has ("APPENDIX"):
        print ("Removing section that is not helpful for embedding models:", text[:100])  # Print the first 100 characters of the removed section for debugging
        return ""  # Remove table of contents and index sections as they are not helpful for embedding models
    if text.has ("\n"):
        lines = text.split("\n")
        cleaned_lines = []
        for line in lines:
            line = line.strip()
            if len(line) < 5:  # Remove very short lines that are likely headers/footers
                continue
            if line.isdigit():  # Remove lines that are just numbers (page numbers)
                continue
            cleaned_lines.append(line)
        return " ".join(cleaned_lines)
def chunk_documents(files_memory, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", "!", "?", " ", ""] # Try to split on paragraphs first, then lines, then sentences, then words, and if all else fails, split at the character level. This helps preserve the semantic structure of the text as much as possible while still creating manageable chunks for embedding models.
    )
    
    all_docs = []
    for filename, pdf in files_memory.items():
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            text = clean_text(text)  # Clean the text to remove weird spacing and newlines
            if not text:
                continue
            
            chunks = splitter.create_documents(
                [text],
                metadatas=[{
                    "source": filename,
                    "page": i
                }]
            )
            all_docs.extend(chunks)
    return all_docs
print (alldocs := chunk_documents(files_memory))
print("Chunking documents with varying chunk sizes and overlaps to find the optimal configuration for embedding models...")



In [ ]:
docs_1000 = chunk_documents(files_memory, 1000, 200)

print("Total chunks:", len(docs_1000))
import matplotlib.pyplot as plt

chunk_lengths = [len(doc.page_content) for doc in docs_1000]

plt.hist(chunk_lengths, bins=20)
plt.title("Chunk Size Distribution (1000)")
plt.xlabel("Chunk length (characters)")
plt.ylabel("Frequency")
plt.show()

Generate embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    print("Warning: HF_TOKEN environment variable not set. If you are using a private model from Hugging Face, please set the HF_TOKEN environment variable to your Hugging Face API token to access the model.")
model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_docs(docs):
    texts = [clean_text(doc.page_content) for doc in docs]
    embeddings = model.encode(texts)
    return embeddings

Store in ChromaDB

In [ ]:
import chromadb

client = chromadb.Client()
collection = client.create_collection(name="pdf_chunks_1000")

def store_in_chroma(docs, embeddings, collection):
    collection.add(
        documents=[doc.page_content for doc in docs],
        embeddings=embeddings.tolist(),
        metadatas=[doc.metadata for doc in docs],
        ids=[str(i) for i in range(len(docs))]
    )

embeddings_1000 = embed_docs(docs_1000)
store_in_chroma(docs_1000, embeddings_1000, collection)
print ("Documents embedded and stored in ChromaDB collection 'pdf_chunks_1000'")

Testing different chunk sizes

In [ ]:
docs_500 = chunk_documents(files_memory, 500, 100)
docs_1000 = chunk_documents(files_memory, 1000, 200)
docs_2000 = chunk_documents(files_memory, 2000, 400)
emb_500 = embed_docs(docs_500)
emb_1000 = embed_docs(docs_1000)
emb_2000 = embed_docs(docs_2000)

col_500 = client.create_collection("chunks_500")
col_1000 = client.create_collection("chunks_1000")
col_2000 = client.create_collection("chunks_2000")

store_in_chroma(docs_500, emb_500, col_500)
store_in_chroma(docs_1000, emb_1000, col_1000)
store_in_chroma(docs_2000, emb_2000, col_2000)
print(
    "Documents embedded and stored in ChromaDB collections with different chunking configurations. You can now query the collections to compare the results and determine the optimal chunk size and overlap for your specific use case."
)

def query_collection(collection, query, model):
    query_embedding = model.encode([query]).tolist()

    all_results = {}
    print ("Querying collection with query:", query, "and model:", model)
    for file in files_memory.keys():

        results = collection.query(
            query_embeddings=query_embedding, n_results=5, where={"source": file}
        )

        all_results[file] = results["documents"]

    return all_results

## Testing queries!

In [ ]:
queries = [
    "What is the main topic of the document?",
    "Explain the key findings",
    "What conclusions are drawn?"
    "List the main sections of the document and their key points."
    "If you can only read 1 chunk from the document, which chunk would you read to get the most important information and why?" # Experimental query to see if the chunking strategy helps the model identify the most important chunk of the document based on the query.
]

for q in queries:
    print("\nQUERY:", q)

    print("\n--- 500 ---")
    print(query_collection(col_500, q, model))

    print("\n--- 1000 ---")
    print(query_collection(col_1000, q, model))

    print("\n--- 2000 ---")
    print(query_collection(col_2000, q, model))

## RAG Pipeline Implementation

In [ ]:
# We need to give the system prompt to the model to answer the question based on the retrieved chunk. We will use a simple prompt that instructs the model to answer the question based on the provided chunk of text. We will also include an instruction to be concise in the answer.
system_prompt = "You are a helpful assistant. Answer the question based on the provided chunk of text. Only answer based on the sources provided, and cite sources."
from openai import OpenAI
if not os.getenv("SATORI_API_KEY"):
    raise ValueError("SATORI_API_KEY environment variable not set. Please set it to your OpenAI API key to access the model.")
client = OpenAI(
    api_key=os.getenv("SATORI_API_KEY"),
    base_url="https://synapse.sergiomathurin.com/v1" 
)

def answer_query_with_gpt(query, retrieved_chunks):
    # Combine the retrieved chunks into a single context string, including source information for each chunk
    context = ""
    for file, chunks in retrieved_chunks.items():
        for chunk in chunks:
            context += f"Source: {file}\n{chunk}\n\n"

    # Create the prompt for the model
    prompt = f"{system_prompt}\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    # Call the OpenAI API to get the answer
    response = client.chat.completions.create(
        model="OpenAI GPT-OSS 20B", # Starting with the cheapest model to test the pipeline. We can experiment with different models to see how it affects the quality of the answers.
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}],
        max_tokens=300,
        temperature=0,
    )
    
    return response

print 